In [1]:
from datetime import datetime

import numpy as np
import pandas as pd
import rasterio as rio
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
import geopandas as gpd
from concurrent.futures import ThreadPoolExecutor

In [2]:
MODEL_NAME = f"XGB_v1_{datetime.now().timestamp()}"
MODEL_NAME

'XGB_v1_1787083166.71639'

In [15]:
DATA_PREFIX = "../data"
OUTPUT_PREFIX = "../output"

TRAIN_PARQUET = f"{OUTPUT_PREFIX}/train_only_biomass.parquet"
TEST_PARQUET = f"{OUTPUT_PREFIX}/test_only_biomass.parquet"
SUBMISSION_CSV = f"{DATA_PREFIX}/sample_submission.csv"

In [4]:
PREDICTORS = ["agb"]

LABEL = "biomass"

In [5]:
train_df = gpd.read_parquet(TRAIN_PARQUET)
train_df

,x,y,year,biomass,tile_id,geometry
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217)
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357)
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368)
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331)
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668)
...,...,...,...,...,...,...
5165029,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312)
5165030,3111624.7416963745,2243883.9191588927,2019,60.487827,035805,POINT (-4.70571 42.23261)
5165031,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345)
5165032,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471)


In [6]:
tile_ids = train_df["tile_id"].unique()

def run_per_tile(tile_id):
    tile_sample = train_df[train_df["tile_id"] == tile_id]
    coords = [coord for coord in zip(tile_sample.geometry.x, tile_sample.geometry.y)]
    years = tile_sample["year"].unique()

    for year in years:
        print(f"Run {tile_id} {year}")
        with rio.open(
            f"https://storage.googleapis.com/gee-ramiqcom-s4g-bucket/opengeohub_summerschool_2026/ctrees_biomass/tile_{tile_id}_{year}.tif"
        ) as src:
            train_df.loc[
                (train_df["tile_id"] == tile_id) & (train_df["year"] == year), "agb"
            ] = [data for data in src.sample(coords)]

with ThreadPoolExecutor(8) as executor:
    jobs = []
    for tile_id in tile_ids:
        jobs.append(executor.submit(run_per_tile, tile_id))
    for job in jobs:
        try:
            job.result()
        except Exception as e:
            print(f"Error: {e}")

train_df

Run 036385 2019
Run 026261 2021
Run 030430 2019
Run 039974 2019
Run 086895 2016
Run 051927 2019
Run 038052 2020
Run 038510 2019
Run 045997 2019
Run 088988 2016
Run 085676 2016
Run 034493 2019
Run 040151 2019
Run 083291 2016
Run 029528 2019
Run 034594 2019
Run 042237 2019
Run 045356 2019
Run 045312 2019
Run 026863 2021
Run 092872 2016
Run 041342 2019
Run 041182 2019
Run 098252 2016
Run 051637 2019
Run 040307 2019
Run 061154 2020
Run 045647 2019
Run 044463 2019
Run 027166 2021
Run 035985 2019
Run 052828 2019
Run 028643 2021
Run 077292 2016
Run 040578 2019
Run 027461 2021
Run 074300 2016
Run 086897 2016
Run 042360 2019
Run 089569 2016
Run 090471 2016
Run 096461 2016
Run 043592 2019
Run 028651 2021
Run 037013 2019
Run 093165 2016
Run 088705 2016
Run 093179 2016
Run 038493 2019
Run 044421 2019
Run 038501 2019
Run 061122 2020
Run 086583 2016
Run 048900 2019
Run 048617 2019
Run 046294 2019
Run 030599 2019
Run 059035 2020
Run 050431 2019
Run 059655 2020
Run 086276 2016
Run 088963 2016
Run 0364

,x,y,year,biomass,tile_id,geometry,agb
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217),10.400000
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357),3.500000
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368),17.799999
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331),3.200000
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668),17.100000
...,...,...,...,...,...,...,...
5165029,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312),8.700000
5165030,3111624.7416963745,2243883.9191588927,2019,60.487827,035805,POINT (-4.70571 42.23261),3.800000
5165031,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345),0.500000
5165032,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471),0.800000


In [7]:
# split train and test data
train, test = train_test_split(train_df, test_size=0.3)

In [8]:
MODEL_NAME = f"XGB_v1_{datetime.now().timestamp()}"
model = XGBRegressor(n_estimators=500)
model.fit(train[PREDICTORS], train[LABEL])

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [9]:
total = sum(model.feature_importances_)
print("Feature importance")
pd.Series(dict(zip(model.feature_names_in_, model.feature_importances_ / total * 100)))

Feature importance


agb    100.0
dtype: float32

In [10]:
test_apply = model.predict(test[PREDICTORS])
r2 = np.corrcoef(test[LABEL], test_apply)[0, 1] ** 2
mae = mean_absolute_error(test[LABEL], test_apply)
print(f"R^2={r2}", f"MAE={mae}")

R^2=0.30389214698191214 MAE=31.56534286460224


In [16]:
test_df = gpd.read_parquet(TEST_PARQUET)
test_df

,row_id,tile_id,x,y,year,geometry
0,028631_3040064_2251003,028631,3040064.7416963745,2251003.9191588927,2021,POINT (-5.57242 42.1662)
1,028631_3041214_2252063,028631,3041214.7416963745,2252063.9191588927,2021,POINT (-5.56121 42.17768)
2,028631_3041184_2251693,028631,3041184.7416963745,2251693.9191588927,2021,POINT (-5.56071 42.17436)
3,028631_3039864_2250783,028631,3039864.7416963745,2250783.9191588927,2021,POINT (-5.57428 42.16389)
4,028631_3040224_2251483,028631,3040224.7416963745,2251483.9191588927,2021,POINT (-5.57163 42.17073)
...,...,...,...,...,...,...
45014,061751_3374644_2043173,061751,3374644.7416963745,2043173.9191588927,2020,POINT (-1.21909 40.85472)
45015,061751_3374594_2043203,061751,3374594.7416963745,2043203.9191588927,2020,POINT (-1.21972 40.85492)
45016,061751_3374624_2043173,061751,3374624.7416963745,2043173.9191588927,2020,POINT (-1.21932 40.85469)
45017,061751_3374614_2043223,061751,3374614.7416963745,2043223.9191588927,2020,POINT (-1.21952 40.85512)


In [61]:
submission_df = pd.read_csv(SUBMISSION_CSV)
submission_df

,Id,Expected
0,028631_3040064_2251003,0.0
1,028631_3041214_2252063,0.0
2,028631_3041184_2251693,0.0
3,028631_3039864_2250783,0.0
4,028631_3040224_2251483,0.0
...,...,...
45014,061751_3374644_2043173,0.0
45015,061751_3374594_2043203,0.0
45016,061751_3374624_2043173,0.0
45017,061751_3374614_2043223,0.0


In [64]:
submission_df.loc[test_df_mask, "Expected"] = model.predict(test_df[NEW_PREDICTORS])
submission_df.loc[~test_df_mask, "Expected"] = 0
submission_df

,Id,Expected
0,028631_3040064_2251003,216.954559
1,028631_3041214_2252063,215.189774
2,028631_3041184_2251693,257.339050
3,028631_3039864_2250783,218.816223
4,028631_3040224_2251483,239.833191
...,...,...
45014,061751_3374644_2043173,49.675964
45015,061751_3374594_2043203,45.027897
45016,061751_3374624_2043173,87.220787
45017,061751_3374614_2043223,136.897537


In [65]:
RESULT_CSV = f"{OUTPUT_PREFIX}/results_{MODEL_NAME}.csv"
submission_df.to_csv(RESULT_CSV, index=False)

MODEL_NAME

'XGB_v1_1787050491.38527'